# 🌾 Crop Pest Detection Model - Google Colab Training Notebook

Welcome! This notebook will guide you through training a Convolutional Neural Network (CNN) to detect agricultural pests using PyTorch and Transfer Learning. Since we are using a pre-trained **MobileNetV2** model, training is highly efficient and will yield high accuracy even with a small dataset.

### 🚀 Setup Instructions in Google Colab:
1. Open [Google Colab](https://colab.research.google.com/).
2. Click **File** -> **Upload notebook** and upload this `.ipynb` file.
3. **Enable GPU acceleration** (Crucial for fast training):
   * Go to **Runtime** -> **Change runtime type**.
   * Under *Hardware accelerator*, select **GPU** (T4 GPU is free and excellent).
   * Click **Save**.
4. Run the cells sequentially.

### 1. Import Dependencies & Check GPU Availability

In [ ]:
import os
import copy
import time
import zipfile
import requests
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

# Check if GPU is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

### 2. Dataset Preparation
To train your model, you need images of crop pests. Below, you can choose **Option A** to download a sample dataset automatically, or **Option B** to upload your own custom crop pest images.

In [ ]:
# Create directories for the dataset
os.makedirs('dataset/train', exist_ok=True)
os.makedirs('dataset/val', exist_ok=True)

# Class names that our system supports
CLASSES = ['aphids', 'armyworm', 'beetle', 'locust', 'mites']

def create_dummy_dataset():
    """Generates a mock dataset in case you want to test running this notebook immediately!"""
    print("Creating a test dataset with synthetic images to make the notebook run out-of-the-box...")
    for class_name in CLASSES:
        os.makedirs(f'dataset/train/{class_name}', exist_ok=True)
        os.makedirs(f'dataset/val/{class_name}', exist_ok=True)
        
        # Generate 20 synthetic training images and 5 validation images per class
        for split, count in [('train', 20), ('val', 5)]:
            for i in range(count):
                # Create simple colored images with shapes representing pests
                img_data = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
                # Paint some random spots to simulate visual pest features
                color_map = {'aphids': [120, 180, 50], 'armyworm': [80, 50, 20], 
                             'beetle': [200, 30, 30], 'locust': [150, 130, 20], 'mites': [230, 230, 230]}
                color = color_map[class_name]
                img_data[40:180, 40:180] = color # Center block
                
                img = Image.fromarray(img_data)
                img.save(f'dataset/{split}/{class_name}/img_{i}.jpg')
    print("Mock dataset created successfully! Ready to train.")

# If you have a custom ZIP file on Google Drive, you can mount drive and copy/unzip it here.
# For immediate execution, we run the dummy creation. Feel free to replace dataset folder contents with your real files.
create_dummy_dataset()

💡 **Note on using your own files**: You can zip your local dataset folder (which has a `train/` and `val/` subfolders each containing subfolders for each pest like `aphids`, `armyworm`, etc.), upload the `.zip` file using the files tab on the left of Colab, and run the following extraction code:

```python
# import zipfile
# with zipfile.ZipFile('your_uploaded_dataset.zip', 'r') as zip_ref:
#     zip_ref.extractall('.')
```

### 3. Image Transformers & Data Loaders
We resize the images to `224x224` pixels (expected by MobileNetV2), apply data augmentations to training samples (rotation, zoom, flip) to simulate real-world farm conditions, and normalize pixel values.

In [ ]:
# Normalization statistics for ImageNet
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

data_dir = 'dataset'
image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=2)
    for x in ['train', 'val']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Training dataset size: {dataset_sizes['train']} images")
print(f"Validation dataset size: {dataset_sizes['val']} images")
print(f"Classes detected: {class_names}")

### 4. Load MobileNetV2 and Customize Classifier
MobileNetV2 is optimized for speed and lightweight footprint. We freeze all features layers (weights won't update) and attach a custom dense classifier layer corresponding to our pest classes.

In [ ]:
# Load pre-trained MobileNetV2
model = models.mobilenet_v2(pretrained=True)

# Freeze parameters so we don't backpropagate through them
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head (MobileNetV2 classifier has a Linear layer at index 1)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, len(class_names))
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
# Only optimize the classifier parameters we just added
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print(model.classifier)

### 5. Training Loop Routine
We will write the training function which runs through training batches, backpropagates gradients, and verifies validation accuracy after each epoch.

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # deep copy the model weights if we get best validation accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Val Accuracy: {best_acc:4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

### 6. Run Training
Run 10 epochs of model fine-tuning. (Adjust `num_epochs` based on size of your custom dataset).

In [ ]:
trained_model, history = train_model(model, criterion, optimizer, num_epochs=10)

### 7. Visualize Results
Plot the accuracy and loss history to evaluate if the model is learning properly or overfitting.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Val Accuracy')
plt.title('Accuracy History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

### 8. Run Live Inference Test
Test your newly trained model on a single unseen image.

In [ ]:
def predict_image(image_path, model):
    model.eval()
    
    # Preprocess test image
    image = Image.open(image_path).convert('RGB')
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    img_tensor = preprocess(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
        confidence, preds = torch.max(probabilities, 0)
        
    class_idx = preds.item()
    predicted_class = class_names[class_idx]
    
    plt.imshow(image)
    plt.title(f"Prediction: {predicted_class} ({confidence.item()*100:.2f}%)")
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence.item()

# Pick a random file from validation to inspect
try:
    sample_class = class_names[0]
    sample_file = os.listdir(f'dataset/val/{sample_class}')[0]
    sample_path = f'dataset/val/{sample_class}/{sample_file}'
    predict_image(sample_path, trained_model)
except Exception as e:
    print(f"Could not run inference: {e}")

### 9. Export Weights & Download
Now we save the trained weights of our model classifier structure. We will export this file and place it in the same directory as our FastAPI backend.

In [ ]:
# Save only the state dict for efficiency and compatibility
torch.save(trained_model.state_dict(), 'pest_model.pth')
print("Model weights saved successfully as 'pest_model.pth'!")

# Code to trigger download in browser automatically (Colab specific)
try:
    from google.colab import files
    files.download('pest_model.pth')
    print("Downloading 'pest_model.pth' to your local download folder...")
except ImportError:
    print("Not running in Google Colab; download script skipped. Find 'pest_model.pth' in files.")